### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [3]:
# !uv add transformers==4.56.2
# !uv pip install --no-deps trl==0.22.2

### Unsloth

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-unsloth-bnb-4bit",
    max_seq_length = 65536, # 64k # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


But with kaiokendev's RoPE scaling of 2.0, it can be magically be extended to 65536!


We now add LoRA adapters so we only need to update a small amount of parameters!

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.9.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Qwen-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Qwen-3 renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [6]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [ ]:
# tokenizer

In [7]:
from datasets import load_dataset

# dataset = load_dataset("mlabonne/FineTome-100k", split = "train")
dataset = load_dataset("hiyouga/glaive-function-calling-v2-sharegpt", split = "train", cache_dir="../data/")

README.md:   0%|          | 0.00/433 [00:00<?, ?B/s]

glaive_toolcall.json:   0%|          | 0.00/251M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100563 [00:00<?, ? examples/s]

In [8]:
dataset[0:10]

{'tools': ['[{"name": "get_exchange_rate", "description": "Get the exchange rate between two currencies", "parameters": {"type": "object", "properties": {"base_currency": {"type": "string", "description": "The currency to convert from"}, "target_currency": {"type": "string", "description": "The currency to convert to"}}, "required": ["base_currency", "target_currency"]}}]',
  '[{"name": "get_news_headlines", "description": "Get the latest news headlines", "parameters": {"type": "object", "properties": {"country": {"type": "string", "description": "The country for which to fetch news"}}, "required": ["country"]}}]',
  '[{"name": "generate_password", "description": "Generate a random password", "parameters": {"type": "object", "properties": {"length": {"type": "integer", "description": "The length of the password"}, "include_symbols": {"type": "boolean", "description": "Whether to include symbols in the password"}}, "required": ["length"]}}, {"name": "create_task", "description": "Create

**References used for the data-processing pipeline**

- [Qwen 2.5 official tool-calling docs](https://qwen.readthedocs.io/en/latest/framework/Function_call.html) — canonical schema for `tool_calls` and tool responses
- [Qwen 2.5 Instruct `tokenizer_config.json`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/raw/main/tokenizer_config.json) — the exact `chat_template` Jinja string the model was trained with (we matched this verbatim)
- [Qwen 2.5-1.5B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) — base model reference
- [Glaive function calling v2 (ShareGPT)](https://huggingface.co/datasets/hiyouga/glaive-function-calling-v2-sharegpt) — source dataset
- [Unsloth `standardize_data_formats` source](https://github.com/unslothai/unsloth/blob/main/unsloth_zoo/dataset_utils.py) — the function that rejects `function_call`/`observation` roles
- [Hugging Face `apply_chat_template` docs](https://huggingface.co/docs/transformers/main/en/chat_templating#advanced-tool-use) — `tools=` kwarg usage


We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

**Custom preprocessing for Glaive function-calling dataset**

The `hiyouga/glaive-function-calling-v2-sharegpt` dataset uses roles `human`, `gpt`, `function_call`, and `observation` — but `standardize_data_formats` only knows about `system`/`user`/`assistant` aliases. We first convert `function_call` -> assistant message with `tool_calls` and `observation` -> `tool` message with `tool_call_id` (Qwen 2.5 tool-calling format), so the chat template renders them correctly.


In [9]:
import json, uuid

def to_qwen_tool_format(example):
    """Convert glaive-function-calling ShareGPT format to Qwen 2.5 tool-calling format.

    Source order:  human -> function_call -> observation -> gpt(answer)
    Target order:  user -> assistant(tool_calls) -> tool -> assistant(answer)

    We SPLIT the post-observation gpt message so the tool_calls come BEFORE
    the tool response, not after. The SFT model needs to learn to emit
    `<tool_call>...</tool_call>` first, then receive `<tool_response>`, then give
    the final answer — that's the format Qwen 2.5 was trained on.

    The 'tools' field is left as a raw JSON string (we don't pre-parse it here)
    because the `datasets` library cannot infer a single PyArrow schema for the
    heterogeneous tool-list column (each row has different parameter shapes).
    The downstream `formatting_prompts_func` parses it before passing to
    `tokenizer.apply_chat_template(tools=...)`.
    """
    convos = example["conversations"]
    new_convo = []
    pending_tool_call_msgs = []

    for msg in convos:
        role = msg["from"]
        text = msg["value"]
        if role == "human":
            new_convo.append({"role": "user", "content": text})
        elif role == "gpt":
            if pending_tool_call_msgs:
                tool_calls = [m["args"] for m in pending_tool_call_msgs if m["role"] == "tool_call"]
                tool_responses = [
                    (m["call_id"], m["content"])
                    for m in pending_tool_call_msgs
                    if m["role"] == "tool_response"
                ]
                if tool_calls:
                    new_convo.append({
                        "role": "assistant",
                        "content": "",
                        "tool_calls": tool_calls,
                    })
                for call_id, content in tool_responses:
                    new_convo.append({
                        "role": "tool",
                        "tool_call_id": call_id,
                        "content": content,
                    })
                pending_tool_call_msgs = []
            new_convo.append({"role": "assistant", "content": text})
        elif role == "function_call":
            try:
                fc = json.loads(text)
                call_id = f"call_{uuid.uuid4().hex[:24]}"
                arg_str = json.dumps(fc.get("arguments", {}), ensure_ascii=False)
                pending_tool_call_msgs.append({
                    "role": "tool_call",
                    "call_id": call_id,
                    "args": {
                        "id": call_id,
                        "type": "function",
                        "function": {
                            "name": fc.get("name", ""),
                            "arguments": arg_str,
                        },
                    },
                })
            except Exception:
                pass
        elif role == "observation":
            if pending_tool_call_msgs:
                latest = None
                for item in reversed(pending_tool_call_msgs):
                    if item["role"] == "tool_call":
                        latest = item
                        break
                if latest:
                    pending_tool_call_msgs.append({
                        "role": "tool_response",
                        "call_id": latest["call_id"],
                        "content": text,
                    })

    if pending_tool_call_msgs:
        tool_calls = [m["args"] for m in pending_tool_call_msgs if m["role"] == "tool_call"]
        tool_responses = [
            (m["call_id"], m["content"])
            for m in pending_tool_call_msgs
            if m["role"] == "tool_response"
        ]
        if tool_calls:
            new_convo.append({"role": "assistant", "content": "", "tool_calls": tool_calls})
        for call_id, content in tool_responses:
            new_convo.append({"role": "tool", "tool_call_id": call_id, "content": content})

    # Keep 'tools' as a raw JSON string to avoid PyArrow schema-inference failures
    # caused by heterogeneous tool definitions across rows.
    raw_tools_str = example.get("tools", "") or ""
    return {"conversations": new_convo, "tools": raw_tools_str}

# Remove only the original 'conversations' field (tools must be preserved as a string).
dataset = dataset.map(
    to_qwen_tool_format,
    remove_columns=[c for c in dataset.column_names if c != "tools"],
)
print("Sample after conversion:")
print(json.dumps(dataset[0], indent=2, default=str)[:2000])


Map:   0%|          | 0/100563 [00:00<?, ? examples/s]

Sample after conversion:
{
  "tools": "[{\"name\": \"get_exchange_rate\", \"description\": \"Get the exchange rate between two currencies\", \"parameters\": {\"type\": \"object\", \"properties\": {\"base_currency\": {\"type\": \"string\", \"description\": \"The currency to convert from\"}, \"target_currency\": {\"type\": \"string\", \"description\": \"The currency to convert to\"}}, \"required\": [\"base_currency\", \"target_currency\"]}}]",
  "conversations": [
    {
      "content": "Can you book a flight for me from New York to London?",
      "role": "user",
      "tool_call_id": null,
      "tool_calls": null
    },
    {
      "content": "I'm sorry, but I don't have the capability to book flights. My current function allows me to get the exchange rate between two currencies. If you need help with that, feel free to ask!",
      "role": "assistant",
      "tool_call_id": null,
      "tool_calls": null
    }
  ]
}


Let's see how row 100 looks like!

In [10]:
dataset[100]

{'tools': '[{"name": "generate_password", "description": "Generate a random password", "parameters": {"type": "object", "properties": {"length": {"type": "integer", "description": "The length of the password"}}, "required": ["length"]}}, {"name": "check_email", "description": "Check if an email address is valid", "parameters": {"type": "object", "properties": {"email": {"type": "string", "description": "The email address to check"}}, "required": ["email"]}}]',
 'conversations': [{'content': 'Hi, I need a new password. Can you generate one for me?',
   'role': 'user',
   'tool_call_id': None,
   'tool_calls': None},
  {'content': 'Of course, I can help with that. How long would you like your password to be?',
   'role': 'assistant',
   'tool_call_id': None,
   'tool_calls': None},
  {'content': 'I would like it to be 12 characters long.',
   'role': 'user',
   'tool_call_id': None,
   'tool_calls': None},
  {'content': '',
   'role': 'assistant',
   'tool_call_id': None,
   'tool_calls'

In [11]:
dataset[202]

{'tools': '[{"name": "calculate_tip", "description": "Calculate the tip amount based on the bill amount and tip percentage", "parameters": {"type": "object", "properties": {"bill_amount": {"type": "number", "description": "The total bill amount"}, "tip_percentage": {"type": "number", "description": "The tip percentage"}}, "required": ["bill_amount", "tip_percentage"]}}, {"name": "calculate_age", "description": "Calculate the age based on birthdate", "parameters": {"type": "object", "properties": {"birthdate": {"type": "string", "description": "The birthdate of the person"}}, "required": ["birthdate"]}}]',
 'conversations': [{'content': 'Hi, I need help calculating the tip for my bill. The total bill amount is $100 and I want to leave a 15% tip.',
   'role': 'user',
   'tool_call_id': None,
   'tool_calls': None},
  {'content': '',
   'role': 'assistant',
   'tool_call_id': None,
   'tool_calls': [{'function': {'arguments': '{"bill_amount": 100, "tip_percentage": 15}',
      'name': 'ca

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [12]:
def _parse_tools(tools_str):
    """Parse the JSON-string tools column into a list[dict] of OpenAI-envelope tools.
    Empty list if '[]' or unparseable."""
    if not tools_str or not isinstance(tools_str, str):
        return []
    s = tools_str.strip()
    if not s or s == "[]":
        return []
    try:
        parsed = json.loads(s)
    except Exception:
        return []
    if not isinstance(parsed, list):
        return []
    out = []
    for t in parsed:
        if not isinstance(t, dict):
            continue
        if "function" in t and isinstance(t["function"], dict):
            if "type" not in t:
                t["type"] = "function"
            out.append(t)
        elif "name" in t:
            out.append({
                "type": "function",
                "function": {
                    "name": t.get("name", ""),
                    "description": t.get("description", ""),
                    "parameters": t.get("parameters", {"type": "object", "properties": {}}),
                },
            })
    return out

def formatting_prompts_func(examples):
   convos = examples["conversations"]
   tools_batch = [_parse_tools(t) for t in examples.get("tools", [""] * len(convos))]
   texts = [
       tokenizer.apply_chat_template(
           convo,
           tools=(tools if tools else None),
           tokenize=False,
           add_generation_prompt=False,
       )
       for convo, tools in zip(convos, tools_batch)
   ]
   return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)


Map:   0%|          | 0/100563 [00:00<?, ? examples/s]

Let's see how the chat template did!

In [13]:
dataset[102]['text']

'<|im_start|>system\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"type": "function", "function": {"name": "calculate_area", "description": "Calculate the area of a shape", "parameters": {"type": "object", "properties": {"shape": {"type": "string", "description": "The shape to calculate the area for"}, "measurements": {"type": "object", "properties": {"length": {"type": "number", "description": "The length of the shape"}, "width": {"type": "number", "description": "The width of the shape"}}, "required": ["length", "width"]}}, "required": ["shape", "measurements"]}}}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{"name": <function-name>, "arguments": <args-json-object>}\n</tool_call><|im_end|>\n<|im_start|>user\nI need to calculate the area of a rectangle. The length is

**Debug: verify `<tool_call>` is in the rendered training text**

SFT teaches the model by imitation. For the model to learn when to emit a `<tool_call>` block, that block must appear in the rendered training text inside an assistant turn. This cell prints the parsed conversation (with `tool_calls` attached) and the rendered text side by side so we can verify both the preprocessing and the chat template are working correctly. See the [Qwen 2.5 tool-calling docs](https://qwen.readthedocs.io/en/latest/framework/Function_call.html) for the expected format.


In [14]:
# Debug: show the parsed conversation AND the rendered text for row 1
# to verify the <tool_call> block is being attached and rendered correctly.
import json
print("=== Parsed conversations (row 1) ===")
print(json.dumps(dataset[1]["conversations"], indent=2)[:2000])
print()
print("=== Tools (row 1) ===")
print(json.dumps(dataset[1]["tools"], indent=2)[:1500])
print()
print("=== Rendered text (row 1) ===")
print(dataset[1]["text"])


=== Parsed conversations (row 1) ===
[
  {
    "content": "Can you tell me the latest news headlines for the United States?",
    "role": "user",
    "tool_call_id": null,
    "tool_calls": null
  },
  {
    "content": "",
    "role": "assistant",
    "tool_call_id": null,
    "tool_calls": [
      {
        "function": {
          "arguments": "{\"country\": \"United States\"}",
          "name": "get_news_headlines"
        },
        "id": "call_255e7919768f474f92be1255",
        "type": "function"
      }
    ]
  },
  {
    "content": "{\"headlines\": [\"Biden announces new vaccine mandates\", \"Hurricane Ida devastates Louisiana\", \"Apple unveils new iPhone\", \"NASA's Perseverance rover collects first Mars rock sample\"]}",
    "role": "tool",
    "tool_call_id": "call_255e7919768f474f92be1255",
    "tool_calls": null
  },
  {
    "content": "Here are the latest news headlines for the United States:\n1. Biden announces new vaccine mandates\n2. Hurricane Ida devastates Louisiana\

In [15]:
print(dataset[1]['text'])

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_news_headlines", "description": "Get the latest news headlines", "parameters": {"type": "object", "properties": {"country": {"type": "string", "description": "The country for which to fetch news"}}, "required": ["country"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
Can you tell me the latest news headlines for the United States?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_news_headlines", "arguments": {"country": "United States"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
{"headlines": ["Biden announces new vaccine mandates", "Hurricane Ida 

In [16]:
print(dataset[22]['text'])

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_books", "description": "Search for books based on title, author, or genre", "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "The title of the book"}, "author": {"type": "string", "description": "The author of the book"}, "genre": {"type": "string", "description": "The genre of the book"}}, "required": []}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
I am looking for a book but I can't remember the title. The author's name is George Orwell.<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "search_books", "arguments": {"auth

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [17]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100563 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [18]:
# from unsloth.chat_templates import train_on_responses_only
# trainer = train_on_responses_only(
#     trainer,
#     instruction_part = "<|im_start|>user\n",
#     response_part = "<|im_start|>assistant\n",
# )

Map (num_proc=2):   0%|          | 0/100563 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [19]:
tokenizer.decode(trainer.train_dataset[104]["input_ids"])

'<|im_start|>user\nWhat PHP code can I use to create a letter to a retired teacher expressing gratitude and sharing how their lessons influenced my current career path? The teacher made a significant impact on my life.<|im_end|>\n<|im_start|>assistant\nHere\'s a sample PHP code that can help you create a letter to a retired teacher expressing gratitude and sharing how their lessons influenced your current career path:\n```\nphp\n// Teacher\'s name\n$teacherName = "Mrs. Johnson";\n// Your name\n$yourName = "John";\n// Your current job\n$currentJob = "Software Engineer";\n// Letter content\n$letter = "Dear $teacherName,nnI hope this letter finds you well. I wanted to take a moment to express my deep gratitude for the impact you had on my life as a student. Your dedication and passion for teaching inspired me to pursue my dreams and helped shape my career path.nnYour lessons on computer science and programming were particularly influential in my decision to become a software engineer. You

Now let's print the masked out example - you should see only the answer is present:

In [21]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                              Of course, I can help with that. How long would you like your password to be?<|im_end|>\n                    <tool_call>\n{"name": "generate_password", "arguments": {"length": 12}}\n</tool_call><|im_end|>\n                                 Here is your new password: aB3#fG6&kL9@. Please make sure to save it in a secure location.<|im_end|>\n'

In [22]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
1.506 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,563 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.552300
2,1.737300
3,1.770900
4,1.561000
5,1.960800
6,1.378700
7,1.474800
8,1.028600
9,1.016500
10,1.117000


In [24]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

264.0897 seconds used for training.
4.4 minutes used for training.
Peak reserved memory = 3.66 GB.
Peak reserved memory for training = 2.154 GB.
Peak reserved memory % of max memory = 25.132 %.
Peak reserved memory for training % of max memory = 14.791 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [31]:
test_user_message = """
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "call_vasim", "description": "Call Mr. Vasim", "parameters": {}}}
</tools>


I want to call vasim?
"""

messages = [
    {"role" : "user", "content" : test_user_message}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 100, # Increase for longer outputs!
    temperature = 0.0, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

To call Mr. Vasim, you can use the function `call_vasim`. Here's an example of how you can use it:
```python
result = call_vasim("Mr. Vasim")
print(result)
```
This will call Mr. Vasim and print the result. Make sure you have the necessary permissions to call external functions. If you're using a different programming language or environment, you may need to adjust the syntax accordingly. If you're using a different function


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [32]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

('qwen_lora/tokenizer_config.json',
 'qwen_lora/special_tokens_map.json',
 'qwen_lora/chat_template.jinja',
 'qwen_lora/vocab.json',
 'qwen_lora/merges.txt',
 'qwen_lora/added_tokens.json',
 'qwen_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!